# Homology-stratified evaluation

Performance on the error-free 300 bp test set stratified by sequence identity to the training and validation data
(DeepCDS N's training data), at the protein level (labeled CDS fragments, MMseqs2 protein search) and the nucleotide
level (complete test sequences, MMseqs2 nucleotide search). Reads the result tables written by `run_audit.sh`.  
Population: the 212 test genomes using translation table 11. Error bars / ±: genome-level cluster bootstrap SE (1,000 replicates, seed 42).

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, ".")       # benchmark_functions.py
sys.path.insert(0, "../..")   # plot_config.py in src_dev/benchmark/eval
import benchmark_functions as bf
from plot_config import FONT_SIZES, MODEL_COLORS, MODEL_DISPLAY_NAMES, MODEL_MARKERS

plt.rcParams.update(FONT_SIZES)
pd.set_option("display.width", 200)

## Settings

In [ ]:
RESULTS_DIR = "/tmp/nrt204/FragmentPredictor/data/processed_data/homology_audit"   # tables from run_audit.sh
OUTPUT_DIR = RESULTS_DIR      # figures/ and tables/ are written next to the result tables
SAVE_OUTPUTS = True
EVALUE_CUTOFF = 10           # 10: main analysis (Figure 5); 1e-3: Supplementary Figure A9 (hits with E > 1e-3 count as no hit)
N_BOOTSTRAP, SEED = 1000, 42

TOOLS = {"deepcds_n": "deep_cds_no_errors", "metaprodigal": "prodigal", "fgs_complete": "fgs_complete"}  # -> plot_config keys
NAMES = {t: MODEL_DISPLAY_NAMES[k] for t, k in TOOLS.items()}
COLORS = {t: MODEL_COLORS[k] for t, k in TOOLS.items()}
MARKERS = {t: MODEL_MARKERS[k] for t, k in TOOLS.items()}

BIN_ORDER = ["no hit", "<30%", "30-50%", "50-70%", "70-90%", ">=90%"]
BIN_LABELS = {"no hit": "No hit", "<30%": "<30%", "30-50%": "30–50%", "50-70%": "50–70%", "70-90%": "70–90%", ">=90%": "≥90%"}
NT_BINS = ["no hit", "<80%", "80-90%", "90-95%", "95-99%", ">=99%"]
NT_EDGES = [0, 0.8, 0.9, 0.95, 0.99, 1.0001]

## Load data

In [ ]:
frag = pd.read_csv(f"{RESULTS_DIR}/per_fragment.tsv.gz", sep="\t")
frag.loc[frag["evalue"] > EVALUE_CUTOFF, "bin"] = "no hit"
df = frag[frag["translation_table"] == 11].copy()
GENOMES = sorted(df["genome"].unique())
df["homology"] = np.select([df["bin"] == "no hit", df["bin"].isin(BIN_ORDER[2:])], ["no hit", "hit >=30%"], "other")

nt = pd.read_csv(f"{RESULTS_DIR}/per_read_nt.tsv.gz", sep="\t")
nt = nt[nt["genome"].isin(GENOMES)]
nt = nt.merge(pd.read_csv(f"{RESULTS_DIR}/per_read_counts_nt.tsv.gz", sep="\t"), on=["genome", "read_name"], how="left")
nt = nt.merge(pd.read_csv(f"{RESULTS_DIR}/read_rna_annotation.tsv.gz", sep="\t", keep_default_na=False),
              on=["genome", "read_name"], how="left", validate="1:1")
count_cols = [f"{t}_{k}" for t in TOOLS for k in ("tp", "fp", "fn")]
nt[count_cols] = nt[count_cols].fillna(0).astype(int)
hit = nt["best_evalue"] <= EVALUE_CUTOFF
nt["nt_bin"] = np.where(hit, pd.cut(nt["best_fident"], NT_EDGES, right=False, labels=NT_BINS[1:]).astype(str), "no hit")

print(f"{len(GENOMES)} genomes, E <= {EVALUE_CUTOFF:g}: {len(df):,} test CDS fragments, {len(nt):,} test sequences")

## Helper functions

In [ ]:
def recall_by_group(sub, group_col, groups):
    """Pooled sensitivity per tool and group, with genome-level cluster bootstrap SE (all GENOMES in the pool)."""
    rng = np.random.default_rng(SEED)
    rows = []
    for tool in TOOLS:
        for grp in groups:
            g = sub[sub[group_col] == grp].groupby("genome")[f"{tool}_tp"].agg(["sum", "count"]).reindex(GENOMES, fill_value=0)
            counts = {a: {"tp": r["sum"], "fp": 0, "fn": r["count"] - r["sum"]} for a, r in g.iterrows()}
            se = bf.bootstrap_pooled_std(counts, n_boot=N_BOOTSTRAP, rng=rng)["recall"]
            rows.append({"tool": tool, "group": grp, "recall": g["sum"].sum() / g["count"].sum(), "se": se, "n": g["count"].sum()})
    return pd.DataFrame(rows)


def nt_metrics(sub, label):
    """Pooled precision, sensitivity and F1 per tool for a set of test sequences, with bootstrap SE."""
    out = []
    for tool in TOOLS:
        g = sub.groupby("genome")[[f"{tool}_tp", f"{tool}_fp", f"{tool}_fn"]].sum().reindex(GENOMES, fill_value=0)
        counts = {a: {"tp": r.iloc[0], "fp": r.iloc[1], "fn": r.iloc[2]} for a, r in g.iterrows()}
        point = bf.pooled_scores(*(np.array([g.iloc[:, i].sum()], float) for i in range(3)))
        se = bf.bootstrap_pooled_std(counts, n_boot=N_BOOTSTRAP, rng=np.random.default_rng(SEED))
        out.append({"group": label, "tool": NAMES[tool], **{k: float(v[0]) for k, v in point.items()},
                    **{f"{k}_se": v for k, v in se.items()}})
    return out


def with_se(long, value, se, index, columns, order):
    """Wide table of "value ± SE" strings."""
    s = long.assign(v=[f"{a:.3f} ± {b:.3f}" for a, b in zip(long[value], long[se])])
    return s.pivot(index=index, columns=columns, values="v")[order]


def save_fig(fig, name):
    if SAVE_OUTPUTS:
        os.makedirs(f"{OUTPUT_DIR}/figures", exist_ok=True)
        fig.savefig(f"{OUTPUT_DIR}/figures/{name}.pdf", bbox_inches="tight")
        fig.savefig(f"{OUTPUT_DIR}/figures/{name}.png", dpi=600, bbox_inches="tight")


def save_table(table, name, latex=None):
    if SAVE_OUTPUTS:
        os.makedirs(f"{OUTPUT_DIR}/tables", exist_ok=True)
        table.to_csv(f"{OUTPUT_DIR}/tables/{name}.csv")
        if latex is not None:
            with open(f"{OUTPUT_DIR}/tables/{name}.tex", "w") as f:
                f.write(latex.to_latex(escape=False, column_format="l" + "c" * latex.shape[1]))


def style_axes(ax):
    ax.yaxis.grid(True, color="#dddddd", linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

## 1. Protein level

Share of test CDS fragments per identity bin of their best hit, sensitivity per bin, and fragment length
per bin (median and quartiles, amino acids).

In [ ]:
dist = df["bin"].value_counts().reindex(BIN_ORDER).to_frame("fragments")
dist["fraction"] = dist["fragments"] / len(df)
display(dist.style.format({"fragments": "{:,}", "fraction": "{:.4f}"}))

prot = recall_by_group(df, "bin", BIN_ORDER)
display(with_se(prot.assign(tool=prot["tool"].map(NAMES)), "recall", "se", "tool", "group", BIN_ORDER))

length = pd.concat([df.assign(group=df["bin"]), df[df["bin"] != "no hit"].assign(group="any hit")])
length = length.groupby("group")["aa_len"].describe()[["count", "25%", "50%", "75%"]].reindex(BIN_ORDER + ["any hit"])
display(length)
save_table(dist, "protein_identity_distribution")
save_table(prot, "protein_sensitivity_per_bin")
save_table(length, "protein_fragment_length_per_bin")

## 2. Nucleotide level

Every test sequence, coding or not, binned by the nucleotide identity of its best hit among the training and
validation sequences. Precision, sensitivity and F1 per bin.

In [ ]:
summary_nt = nt["nt_bin"].value_counts().reindex(NT_BINS).to_frame("reads")
summary_nt["fraction"] = summary_nt["reads"] / len(nt)
display(summary_nt.style.format({"reads": "{:,}", "fraction": "{:.4f}"}))

nt_perf = pd.DataFrame([r for b in NT_BINS for r in nt_metrics(nt[nt["nt_bin"] == b], b)])
for metric in ("f1_score", "precision", "recall"):
    wide = with_se(nt_perf, metric, f"{metric}_se", "tool", "group", NT_BINS).reindex(list(NAMES.values()))
    print(metric)
    display(wide)
save_table(summary_nt, "nt_identity_distribution")
save_table(nt_perf, "nt_performance_per_bin")

## 3. rRNA and tRNA genes

**Annotation categories.** A test sequence is *Labeled CDS* if it contains a labeled CDS fragment, otherwise
*rRNA* or *tRNA* if such a gene (either strand) overlaps it by > 60 bp (`09_annotate_reads_rna.py`), otherwise *Other*.

**False positives.** FP CDSs on the sequences with a >= 90% identical hit, by category.

**Genome share.** Share of genome positions covered by annotated rRNA or tRNA genes, all
1,125 genomes (`10_rna_gene_coverage.py`).

In [ ]:
NT_CATS = ["Labeled CDS", "rRNA", "tRNA", "Other"]
labeled = (nt["deepcds_n_tp"] + nt["deepcds_n_fn"]) > 0
nt["feature_class"] = np.select([labeled, nt["rna_gene"] == "rRNA", nt["rna_gene"] == "tRNA"], NT_CATS[:3], "Other")
composition = pd.crosstab(nt["nt_bin"], nt["feature_class"], normalize="index").reindex(index=NT_BINS, columns=NT_CATS).fillna(0)
display(composition.round(3))
save_table(composition, "nt_annotation_categories")

high = nt[nt["nt_bin"].isin(["90-95%", "95-99%", ">=99%"])]
fp = pd.DataFrame({NAMES[t]: high.groupby("feature_class")[f"{t}_fp"].sum() for t in ["metaprodigal", "fgs_complete", "deepcds_n"]})
fp = fp.reindex(NT_CATS).fillna(0).astype(int)
fp_table = pd.DataFrame({"FPs on sequences with >= 90% identity": fp.sum(), "overlapping rRNA or tRNA genes": fp.loc["rRNA"] + fp.loc["tRNA"],
                         "overlapping rRNA genes": fp.loc["rRNA"], "overlapping tRNA genes": fp.loc["tRNA"]}).T
latex = fp_table.apply(lambda col: [f"{v:,}" if i == 0 else f"{v:,} ({100 * v / col.iloc[0]:.1f}\\%)" for i, v in enumerate(col)])
latex.index = ["FPs on sequences with $\\geq$90\\% identity", "\\quad overlapping rRNA or tRNA genes",
               "\\quad\\quad overlapping rRNA genes", "\\quad\\quad overlapping tRNA genes"]
print(f"{len(high):,} test sequences with a >= 90% identical hit")
display(fp_table)
save_table(fp_table, "fp_rrna_trna_supplementary", latex=latex)

cov = pd.read_csv(f"{RESULTS_DIR}/rna_gene_coverage_per_genome.tsv", sep="\t")
cov_summary = pd.concat({**{p: cov[cov["partition"] == p] for p in ("train", "val", "test")}, "all": cov})
cov_summary = 100 * cov_summary.groupby(level=0, sort=False)[["rRNA_fraction", "tRNA_fraction", "rna_fraction"]].median()
display(cov_summary.round(2).rename(columns=lambda c: f"median {c.replace('_fraction', '').replace('rna', 'rRNA or tRNA')} (%)"))
save_table(cov_summary, "rna_gene_genome_share_median")

fig, ax = plt.subplots(figsize=(9, 5))
v = 100 * cov["rna_fraction"]
ax.hist(v, bins=np.arange(0, 3.05, 0.1), color=COLORS["deepcds_n"], edgecolor="white", linewidth=1, zorder=2)
ax.axvline(v.median(), color="#333333", linestyle="--", linewidth=1.5, zorder=3)
ax.text(0.98, 0.95, f"median {v.median():.2f}%", transform=ax.transAxes, ha="right", va="top", fontsize=16)
ax.set_xlabel("Genome covered by rRNA and tRNA genes (%)")
ax.set_ylabel(f"Genomes (n = {len(cov):,})")
style_axes(ax)
plt.show()
save_fig(fig, "rna_gene_coverage_histogram")

## 4. Figure with the 6 subplots

In [ ]:
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

TOOL_ORDER = ["fgs_complete", "metaprodigal", "deepcds_n"]   # legend order as in the other figures
FEATURE_COLORS = {"Labeled CDS": "#d4d4d0", "tRNA": "#eda100", "rRNA": "#6250d6", "Other": "#e34948"}  # validated (dataviz check)
NT_TICKS = [b.replace(">=", "≥").replace("no hit", "No hit").replace("-", "–") for b in NT_BINS]
NT_TICKS_N = [f"{t} ({int(n):,})" for t, n in zip(NT_TICKS, summary_nt["reads"])]   # reads per bin
plt.rcParams.update({"xtick.labelsize": 18, "ytick.labelsize": 18, "axes.labelsize": 19, "axes.titlesize": 19,
                     "legend.fontsize": 18})   # larger labels for this figure (reset at the end of the cell)
LEN_EDGES = [19, 29, 39, 49, 59, 69, 79, 89, 100]
LEN_LABELS = ["20–29", "30–39", "40–49", "50–59", "60–69", "70–79", "80–89", "90–100"]

df["len10"] = pd.cut(df["aa_len"], LEN_EDGES, labels=LEN_LABELS).astype(str)                   # panel B
by_len = pd.concat([recall_by_group(df[df["homology"] == h], "len10", LEN_LABELS).assign(homology=h)
                    for h in ("no hit", "hit >=30%")])
comp_plot = composition[list(FEATURE_COLORS)].copy()                                             # panel F


def dot_panel(ax, long, groups, ticks, value="recall", se="se", key=lambda t: t, ylim=None, offset=0.16):
    x = np.arange(len(groups))
    for k, tool in enumerate(TOOL_ORDER):
        s = long[long["tool"] == key(tool)].set_index("group").reindex(groups)
        ax.errorbar(x + (k - (len(TOOL_ORDER) - 1) / 2) * offset, s[value], yerr=s[se], fmt=MARKERS[tool], color=COLORS[tool], markersize=8,
                    capsize=3, elinewidth=1.5, linewidth=0, label=NAMES[tool])
    ax.set_xticks(x, ticks)
    if ylim:
        ax.set_ylim(*ylim)
    style_axes(ax)


fig = plt.figure(figsize=(20, 15))
top = GridSpec(1, 2, figure=fig, left=0.055, right=0.985, top=0.96, bottom=0.66, wspace=0.16)
bot = GridSpec(1, 4, figure=fig, left=0.055, right=0.985, top=0.53, bottom=0.25, wspace=0.34)

# A: protein-level recall per identity bin
axA = fig.add_subplot(top[0, 0])
n_bin = prot.groupby("group")["n"].first().reindex(BIN_ORDER)
dot_panel(axA, prot, BIN_ORDER, [f"{BIN_LABELS[b]}\n({n:,})" for b, n in zip(BIN_ORDER, n_bin)], ylim=(0.5, 1.0), offset=0.13)
axA.set_xlabel("Best-hit amino acid sequence identity (CDS fragments)"); axA.set_ylabel("Sensitivity")
axA.set_title("A  Protein level: sensitivity per sequence identity bin", loc="left", fontweight="bold")

# B: no-hit recall by fragment length (dashed: fragments with a >= 30% hit)
axB = fig.add_subplot(top[0, 1])
x = np.arange(len(LEN_LABELS))
for tool in TOOL_ORDER:
    for h, ls, alpha in (("no hit", "-", 1.0), ("hit >=30%", "--", 0.45)):
        s = by_len[(by_len["tool"] == tool) & (by_len["homology"] == h)].set_index("group").reindex(LEN_LABELS)
        axB.errorbar(x, s["recall"], yerr=s["se"] if h == "no hit" else None, color=COLORS[tool], linestyle=ls,
                     marker=MARKERS[tool] if h == "no hit" else None, markersize=7, capsize=3, linewidth=1.8, alpha=alpha)
n_len = by_len[by_len["homology"] == "no hit"].groupby("group")["n"].first().reindex(LEN_LABELS)
axB.set_xticks(x, [f"{l}\n({n:,})" for l, n in zip(LEN_LABELS, n_len)], fontsize=16)
axB.set_ylim(0.4, 1.0); axB.set_xlabel("CDS fragment length (aa)"); axB.set_ylabel("Sensitivity")
axB.set_title("B  Protein level: no-hit sensitivity by fragment length", loc="left", fontweight="bold")
style_axes(axB)
axB.legend(handles=[Line2D([], [], color="0.3", linestyle="-", label="No hit"),
                    Line2D([], [], color="0.3", linestyle="--", alpha=0.6, label="Hit (≥30%)")],
           loc="lower right", frameon=False)

# C-E: nucleotide-level F1, precision, recall
for j, (metric, label) in enumerate([("f1_score", "F1 score"), ("precision", "Precision"), ("recall", "Sensitivity")]):
    ax = fig.add_subplot(bot[0, j])
    dot_panel(ax, nt_perf, NT_BINS, NT_TICKS, value=metric, se=f"{metric}_se", key=lambda t: NAMES[t], ylim=(0.0, 1.0), offset=0.13)
    ax.set_xticklabels(NT_TICKS, rotation=45, ha="right", rotation_mode="anchor")   # read counts only in panel F
    ax.set_ylabel(label)
    ax.set_title(f"{'CDE'[j]}  Nucleotide level: {'F1 score' if j == 0 else 'Sensitivity' if j == 2 else label.capitalize()}", loc="left", fontweight="bold", fontsize=17)

# F: genomic features of the reads per nucleotide bin
axF = fig.add_subplot(bot[0, 3])
xf, bottom = np.arange(len(NT_BINS)), np.zeros(len(NT_BINS))
for cat, color in FEATURE_COLORS.items():
    v = comp_plot.loc[NT_BINS, cat].to_numpy()
    axF.bar(xf, v, bottom=bottom, color=color, width=0.7, edgecolor="white", linewidth=1.5, label=cat)
    bottom += v
axF.set_xticks(xf, NT_TICKS_N, rotation=45, ha="right", rotation_mode="anchor"); axF.set_ylim(0, 1); axF.set_ylabel("Share of test sequences")
axF.set_title("F  Annotation categories", loc="left", fontweight="bold", fontsize=17)
for s in ("top", "right"):
    axF.spines[s].set_visible(False)

fig.text(0.4, 0.1, "Best-hit nucleotide sequence identity", ha="center", fontsize=19)
tool_handles = [Line2D([], [], color=COLORS[t], marker=MARKERS[t], linestyle="", markersize=9, label=NAMES[t]) for t in TOOL_ORDER]
feature_handles = [plt.Rectangle((0, 0), 1, 1, color=c, label=k) for k, c in FEATURE_COLORS.items()]
fig.legend(handles=tool_handles, loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.28, 0.02))
fig.legend(handles=feature_handles, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.78, 0.02))
save_fig(fig, "homology_stratified_performance_E10" if EVALUE_CUTOFF >= 10 else "homology_stratified_performance_E0.001")
plt.show()
plt.rcParams.update(FONT_SIZES)